# 05. Production Monitoring
**Objective:** Monitor the health of the live system, check for model drift, and verify data quality.

**Requires:** `results/predictions/alpha_signals_*.parquet`.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('../').resolve()
sys.path.append(str(PROJECT_ROOT))

from config.settings import config
from quant_alpha.utils import load_parquet
from quant_alpha.monitoring.model_drift import ModelDriftDetector
from quant_alpha.monitoring.data_quality import DataQualityMonitor

%matplotlib inline

## 1. Load Latest Predictions

In [ ]:
pred_dir = config.RESULTS_DIR / 'predictions'
files = sorted(pred_dir.glob('alpha_signals_*.parquet'))

if not files:
    raise FileNotFoundError("No predictions found.")

latest_file = files[-1]
print(f"Loading latest predictions: {latest_file.name}")
preds = load_parquet(latest_file)
preds['date'] = pd.to_datetime(preds['date'])

## 2. Signal Distribution Check
Ensure the alpha distribution hasn't collapsed or shifted drastically.

In [ ]:
plt.figure(figsize=(10, 5))
preds['ensemble_alpha'].hist(bins=50, alpha=0.7)
plt.title('Latest Alpha Distribution')
plt.show()

print(preds['ensemble_alpha'].describe())

## 3. Data Quality Check
Compare incoming data against a reference set (historical master data).

In [ ]:
# Load reference data (e.g., last month of master data)
master = load_parquet(config.CACHE_DIR / 'master_data_with_factors.parquet')
ref_data = master[master['date'] < '2023-01-01'].sample(5000)

monitor = DataQualityMonitor(reference_data=ref_data)
report = monitor.check_incoming_data(preds, data_type='predictions')

print(f"Status: {report['status']}")
if report['issues']:
    print("Issues Found:", report['issues'])